In [1]:
import pandas as pd

# --- Load and prepare data ---
df = pd.read_csv("CAR_data_V2.csv")
df['Activity Start Timestamp'] = pd.to_datetime(df['Activity Start Timestamp'], errors='coerce')
df = df.sort_values(['Contact Session ID', 'Activity Start Timestamp'])


/var/folders/h7/c_5v7d7s0w7d_6fry6l74vyc0000gn/T/ipykernel_8877/518779745.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("CAR_data_V2.csv")


In [2]:
def calculate_adjusted_duration(group):
    group = group.sort_values('Activity Start Timestamp').reset_index(drop=True)
    activities = group['Activity Name'].astype(str).tolist()
    times = group['Activity Start Timestamp'].tolist()

    STOP_EVENTS = {"DisconnectContact1", "DisconnectContact", "DisconnectContact2", "CCB"}
    RESTART_EVENTS = {"LegalServerScreenPop"}

    total_duration = 0
    active = True
    start_time = times[0]

    for i, act in enumerate(activities):
        # Normal stop events
        if act in STOP_EVENTS and active:
            total_duration += (times[i] - start_time).total_seconds() / 60
            active = False

        # Special handling for CallbackRetry
        elif act == "CallbackRetry" and active:
            # Stop 3 rows prior if possible
            stop_index = max(i - 3, 0)
            total_duration += (times[stop_index] - start_time).total_seconds() / 60
            active = False

        # Restart events
        elif act in RESTART_EVENTS and not active:
            start_time = times[i]
            active = True

    # If still active at the end, close the last segment
    if active:
        total_duration += (times[-1] - start_time).total_seconds() / 60

    return total_duration


In [3]:
adjusted = df.groupby('Contact Session ID', group_keys=False).apply(calculate_adjusted_duration)

In [4]:
invalid_tokens = {'', 'nan', 'na', 'none', 'null', 'n/a', 'unknown'}

def compute_stage_durations(group, adjusted_call_duration):
    """
    group: dataframe for one Contact Session ID
    adjusted_call_duration: precomputed value from your old logic
    """

    group = group.sort_values('Activity Start Timestamp').reset_index(drop=True)

    activities = group['Activity Name'].fillna("").astype(str).tolist()
    agents     = group['Agent Name'].fillna("").astype(str).tolist()
    times      = group['Activity Start Timestamp'].tolist()


    # -----------------------------
    # QUEUE TIME — multiple segments
    # -----------------------------
    queue_time = 0
    i = 0
    while i < len(activities):
        if "PreQueue" in activities[i]:
            start = times[i]
            j = i + 1
            while j < len(activities) and not any(
                end_token in activities[j]
                for end_token in ["PlayMOH300s", "QueueMenu1", "DisconnectContact1", "CCB"]
            ):
                j += 1
            if j < len(activities):
                end = times[j]
            else:
                end = times[-1]
            queue_time += (end - start).total_seconds() / 60
            i = j + 1
        else:
            i += 1

    # -----------------------------
    # AGENT TIME — multiple segments
    # -----------------------------
    agent_time = 0

    # Normalize agent names
    norm_agents = [
        a.strip().lower() if isinstance(a, str) else ""
        for a in agents
    ]

    has_agent = any(a not in invalid_tokens for a in norm_agents)

    if has_agent:
        i = 0
        while i < len(norm_agents):
            if norm_agents[i] not in invalid_tokens:
                # start of a consecutive agent segment
                start = times[i]
                j = i + 1
                while j < len(norm_agents) and norm_agents[j] == norm_agents[i]:
                    j += 1
                end = times[j-1]
                agent_time += (end - start).total_seconds() / 60
                i = j
            else:
                i += 1

    # -----------------------------
    # MENU TIME — residual
    # -----------------------------
    menu_time = adjusted_call_duration - agent_time - queue_time
    menu_time = max(menu_time, 0)

    # -----------------------------
    # RETURN
    # -----------------------------
    return pd.Series({
        "Menu_Time_Min": menu_time,
        "Queue_Time_Min": queue_time,
        "Agent_Time_Min": agent_time,
        "Adjusted_Call_Duration_Min": adjusted_call_duration
    })


In [5]:
# confirming agent name calculation includes front desk transfers
# assuming df already loaded and cleaned
df['Activity Name'] = df['Activity Name'].astype(str)
df['Agent Name'] = df['Agent Name'].astype(str)

# Condition: activity contains "frontdesk" (case-insensitive)
frontdesk_mask = df['Activity Name'].str.contains("FrontDeskTransfer", case=False, na=False)

# Rows where activity contains "frontdesk" AND agent name is blank
problem_rows = df[frontdesk_mask & (df['Agent Name'].str.strip() == "")]

# Get the violating Contact Session IDs
violating_ids = problem_rows['Contact Session ID'].unique().tolist()

print("Contact Session IDs where frontdesk appears but agent is missing:")
print(violating_ids)

Contact Session IDs where frontdesk appears but agent is missing:
[]


In [ ]:
# -----------------------------
# 1. NORMALIZE Agent Name
# -----------------------------
df['Agent Name'] = df['Agent Name'].astype('string').str.strip()

# -----------------------------
# 2. AGENT LOGIC (your corrected version)
# -----------------------------
invalid_tokens = {'', 'nan', 'na', 'none', 'null', 'n/a', 'unknown'}

def has_valid_agent(s):
    s = s.astype('string').str.strip().str.lower()
    return (~s.isin(invalid_tokens)).any()

agent_present_by_call = (
    df.groupby('Contact Session ID')['Agent Name']
      .apply(has_valid_agent)
      .rename('Has_Agent')
      .reset_index()
)

# -----------------------------
# 3. SENIOR LOGIC
# -----------------------------
senior_by_call = (
    df.groupby('Contact Session ID')['Activity Name']
      .apply(lambda s: s.astype(str).str.contains("SuburbsOrCityMenu", na=False).any())
      .rename('Is_Senior')
      .reset_index()
)

# -----------------------------
# 4. COMBINE FLAGS
# -----------------------------
call_flags = agent_present_by_call.merge(senior_by_call, on="Contact Session ID", how="outer")

# Make sure the merge key matches dtype with results
call_flags['Contact Session ID'] = call_flags['Contact Session ID'].astype(str)

# -----------------------------
# 5. MERGE adjusted duration into df BEFORE computing results
# -----------------------------
df['Contact Session ID'] = df['Contact Session ID'].astype(str)
adjusted.index = adjusted.index.astype(str)

df = df.merge(
    adjusted.rename("Adjusted_Call_Duration_Min"),
    left_on="Contact Session ID",
    right_index=True,
    how="left"
)

# -----------------------------
# 6. COMPUTE RESULTS
# -----------------------------
results = (
    df.groupby("Contact Session ID")
      .apply(lambda g: compute_stage_durations(
          g,
          adjusted_call_duration=g["Adjusted_Call_Duration_Min"].iloc[0]
      ))
      .reset_index()
)

results['Contact Session ID'] = results['Contact Session ID'].astype(str)

# -----------------------------
# 7. MERGE FLAGS INTO RESULTS
# -----------------------------
results = results.merge(call_flags, on="Contact Session ID", how="left")

# -----------------------------
# 8. DIAGNOSTICS
# -----------------------------
print("\nMerged call_flags sample:")
print(call_flags.head())

print("\nresults with new columns:")
print(results[['Contact Session ID','Has_Agent','Is_Senior']].head())

print("\nCounts:")
print("Has_Agent:", results['Has_Agent'].value_counts(dropna=False))
print("Is_Senior:", results['Is_Senior'].value_counts(dropna=False))


In [ ]:
# -----------------------------
# FINAL METRIC CALCULATIONS
# -----------------------------

# Overall averages
overall_avgs = results[['Menu_Time_Min', 'Queue_Time_Min', 'Agent_Time_Min']].mean()

# Seniors
senior_avgs = results[results['Is_Senior'] == True][['Menu_Time_Min', 'Queue_Time_Min', 'Agent_Time_Min']].mean()

# Non-Seniors
non_senior_avgs = results[results['Is_Senior'] == False][['Menu_Time_Min', 'Queue_Time_Min', 'Agent_Time_Min']].mean()

# Put into a nice summary table
summary = pd.DataFrame({
    'Overall': overall_avgs,
    'Seniors': senior_avgs,
    'Non-Seniors': non_senior_avgs
})

summary.index = ['Average Menu Time', 'Average Queue Time', 'Average Agent Time']
print(summary.round(2))


In [ ]:
results.to_csv("call_stage_durations.csv", index=False)
